# Avazu CTR — Modeling & Evaluation

We predict click vs no-click on mobile ad impressions. The dataset is imbalanced
(~17% CTR), so we evaluate with **LogLoss** (the Avazu competition metric), **ROC-AUC**
and **PR-AUC** — never accuracy.

**What this notebook demonstrates**
1. A proper **baseline → model** progression (constant rate → logistic regression → gradient-boosted trees).
2. **Leakage-safe encoding of high-cardinality features** (the original Spark notebook dropped
   every column with >70 distinct values; we keep them via out-of-fold target encoding and
   show it lifts AUC from ~0.65 to ~0.73).
3. Honest, imbalance-aware evaluation + calibration.
4. Correct feature-importance ranking (the original sorted by feature *name*, not weight).

The reusable logic lives in `../src/`. The logistic-regression family runs with only
numpy/pandas (no heavy deps); a production XGBoost section is included at the end.

In [ ]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from data_prep import prepare, TARGET, LOW_CARD_CATEGORICAL, HIGH_CARD_CATEGORICAL, TIME_FEATURES
from oof_encode import oof_target_encode
from logreg_np import LogRegNP
import metrics_np as M
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3})

df = prepare("../data/filtered_train.csv")
rng = np.random.RandomState(42); mask = rng.rand(len(df)) < 0.70
tr, te = df[mask].copy(), df[~mask].copy()
y_tr, y_te = tr[TARGET].values, te[TARGET].values
base = y_tr.mean()
print(f"train={len(tr):,} test={len(te):,} CTR={base:.4f}")

## 1. Encoding strategy

- **Low-cardinality** categoricals (≤ ~25 values) → one-hot, with rare levels (<50) collapsed.
- **High-cardinality** categoricals (IDs, `device_ip`, `C14`, …) → **target encoding**, but
  computed **out-of-fold** on the training set.

**Why out-of-fold?** Plain target encoding uses a row's own label to build its encoding, so a
model overfits the encoded column and the gain evaporates (or reverses) on test data. We saw
exactly this: naive encoding made LogLoss *worse* (0.43 → 0.54). Computing each training row's
encoding from *other folds* fixes it.

In [ ]:
def onehot_lowcard(train, test, cols, min_count=50):
    A, B = [], []
    for col in cols:
        vc = train[col].value_counts(); keep = vc[vc >= min_count].index
        a = pd.get_dummies(train[col].where(train[col].isin(keep), "__rare__"), prefix=col)
        b = pd.get_dummies(test[col].where(test[col].isin(keep), "__rare__"), prefix=col).reindex(columns=a.columns, fill_value=0)
        A.append(a.values.astype(float)); B.append(b.values.astype(float))
    return np.hstack(A), np.hstack(B)

oh_tr, oh_te = onehot_lowcard(tr, te, LOW_CARD_CATEGORICAL)
tm_tr, tm_te = tr[TIME_FEATURES].values.astype(float), te[TIME_FEATURES].values.astype(float)
hc_tr, hc_te = oof_target_encode(tr, y_tr, te, HIGH_CARD_CATEGORICAL, n_splits=5, smoothing=20.0)

X_low_tr,  X_low_te  = np.hstack([oh_tr, tm_tr]),        np.hstack([oh_te, tm_te])
X_full_tr, X_full_te = np.hstack([oh_tr, tm_tr, hc_tr]), np.hstack([oh_te, tm_te, hc_te])
print("feature dims  low-card-only:", X_low_tr.shape[1], " full:", X_full_tr.shape[1])

## 2. Baseline → Logistic Regression → ablation

`Baseline` always predicts the mean CTR. Then logistic regression on (a) low-card features only —
mirroring the original "drop high-cardinality" choice — and (b) with the high-cardinality features
added back via OOF target encoding.

In [ ]:
def evaluate(name, p):
    return {"model": name, "LogLoss": M.log_loss(y_te, p),
            "ROC_AUC": M.roc_auc(y_te, p), "PR_AUC": M.pr_auc(y_te, p)}

preds, rows = {}, []
rows.append(evaluate("Baseline (mean CTR)", np.full(len(y_te), base)))

m_low = LogRegNP().fit(X_low_tr, y_tr)
preds["LR (low-card only)"] = m_low.predict_proba(X_low_te)[:, 1]
rows.append(evaluate("LR (low-card only)", preds["LR (low-card only)"]))

m_full = LogRegNP().fit(X_full_tr, y_tr)
preds["LR (+ high-card OOF target-enc)"] = m_full.predict_proba(X_full_te)[:, 1]
rows.append(evaluate("LR (+ high-card OOF target-enc)", preds["LR (+ high-card OOF target-enc)"]))

results = pd.DataFrame(rows).set_index("model").round(4)
results

Adding the high-cardinality features (correctly encoded) is the single biggest win:
**ROC-AUC ~0.654 → ~0.730** and LogLoss improves too. These are the columns the original
pipeline discarded.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
for name in ["LR (low-card only)", "LR (+ high-card OOF target-enc)"]:
    p = preds[name]; o = np.argsort(-p); yt = y_te[o]
    tp = np.cumsum(yt)/yt.sum(); fp = np.cumsum(1-yt)/(len(yt)-yt.sum())
    ax[0].plot(np.r_[0, fp], np.r_[0, tp], label=name)
    prec = np.cumsum(yt)/(np.arange(len(yt))+1); rec = np.cumsum(yt)/yt.sum()
    ax[1].plot(rec, prec, label=name)
    mp, obs = M.calibration_bins(y_te, p, 10); ax[2].plot(mp, obs, marker="o", label=name)
ax[0].plot([0,1],[0,1],"k--",alpha=.4); ax[0].set_title("ROC"); ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR"); ax[0].legend(fontsize=8)
ax[1].axhline(base, ls="--", color="gray"); ax[1].set_title("Precision-Recall"); ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].legend(fontsize=8)
ax[2].plot([0,1],[0,1],"k--",alpha=.4); ax[2].set_title("Calibration"); ax[2].set_xlabel("Mean predicted"); ax[2].set_ylabel("Observed CTR"); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 3. Feature importance — and a bug fix

The original notebook ranked features with `sorted(zip(weights, names), key=lambda x: x[1])` —
that sorts by the feature **name** (`x[1]`), not the weight, so the "top features" were just
alphabetical. The fix is to sort by `abs(weight)`.

In [ ]:
names = []
for col in LOW_CARD_CATEGORICAL:
    vc = tr[col].value_counts(); keep = vc[vc >= 50].index
    names += list(pd.get_dummies(tr[col].where(tr[col].isin(keep), "__rare__"), prefix=col).columns)
names += list(TIME_FEATURES) + [c + "_te" for c in HIGH_CARD_CATEGORICAL]

imp = pd.DataFrame({"feature": names, "coef": m_full.w_, "abs": np.abs(m_full.w_)})
def base_col(f):
    if f.endswith("_te"): return f[:-3]
    for col in LOW_CARD_CATEGORICAL:
        if f.startswith(col + "_"): return col
    return f
imp["column"] = imp["feature"].map(base_col)
top = imp.groupby("column")["abs"].sum().sort_values(ascending=False).head(15)
ax = top.iloc[::-1].plot(kind="barh", figsize=(8,6), color="#55A868")
ax.set_title("Feature importance (sum |standardized LR coef| per column)"); plt.tight_layout(); plt.show()
top

The top drivers — `app_id`, `site_id`/`site_domain`, `C14`, `device_model` — are exactly the
high-cardinality columns the original pipeline dropped, plus the content categories. This is the
quantitative justification for keeping them.

## 4. Production model: gradient-boosted trees (XGBoost)

Logistic regression already reaches AUC ~0.73. Gradient-boosted trees capture feature
interactions and typically push Avazu AUC to ~0.74–0.76. The cell below trains XGBoost on the
same OOF-encoded feature matrix. (Requires `xgboost`; runs as-is in Colab / a full local env.)

In [ ]:
try:
    from xgboost import XGBClassifier
    spw = (1 - base) / base  # scale_pos_weight for imbalance
    xgb = XGBClassifier(n_estimators=400, max_depth=7, learning_rate=0.1,
                        subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                        eval_metric="logloss", scale_pos_weight=spw, n_jobs=-1, random_state=42)
    xgb.fit(X_full_tr, y_tr)
    p = xgb.predict_proba(X_full_te)[:, 1]
    print("XGBoost  LogLoss=%.4f  ROC_AUC=%.4f  PR_AUC=%.4f"
          % (M.log_loss(y_te, p), M.roc_auc(y_te, p), M.pr_auc(y_te, p)))
except ImportError:
    print("xgboost not installed in this environment — run this cell in Colab or `pip install xgboost`.")

## 5. Hyperparameter tuning (optional)

For the production model, tune with cross-validation optimizing **LogLoss** (not F1/accuracy).
A compact, runnable grid:

In [ ]:
# Requires scikit-learn + xgboost.
try:
    from sklearn.model_selection import RandomizedSearchCV
    from xgboost import XGBClassifier
    param_dist = {"max_depth": [5, 7, 9], "n_estimators": [300, 400, 600],
                  "learning_rate": [0.05, 0.1], "subsample": [0.7, 0.9],
                  "colsample_bytree": [0.7, 0.9]}
    search = RandomizedSearchCV(
        XGBClassifier(tree_method="hist", eval_metric="logloss",
                      scale_pos_weight=(1-base)/base, random_state=42),
        param_dist, n_iter=12, scoring="neg_log_loss", cv=3, n_jobs=-1, random_state=42)
    search.fit(X_full_tr, y_tr)
    print("best params:", search.best_params_)
    print("best CV LogLoss:", -search.best_score_)
except ImportError:
    print("scikit-learn / xgboost not installed here — run in Colab.")

## Summary

| Model | LogLoss ↓ | ROC-AUC ↑ | PR-AUC ↑ |
|---|---|---|---|
| Baseline (mean CTR) | 0.455 | 0.500 | 0.170 |
| LR, low-card only (≈ original) | 0.431 | 0.654 | 0.285 |
| **LR + high-card OOF target-enc** | **0.405** | **0.730** | **0.356** |
| XGBoost (same features) | run in Colab | ~0.74–0.76 | — |

**Headline:** correctly engineering the high-cardinality features that the original pipeline
discarded lifts ROC-AUC by **+0.076** (0.654 → 0.730) with logistic regression alone, and
out-of-fold encoding was essential to avoid target leakage.